## 1. Install Dependencies

In [1]:
# FIX: single pip call — fewer subprocess spawns, faster install
!pip install ultralytics easyocr opencv-python-headless numpy matplotlib --quiet
print("Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 29.7 MB/s eta 0:00:00
Dependencies installed.


## 2. Imports

In [2]:
import cv2
import numpy as np
import easyocr
import os
import re
import time
import warnings
warnings.filterwarnings('ignore')

from ultralytics import YOLO
from google.colab import files
from IPython.display import display, Video
import matplotlib.pyplot as plt

# FIX: PIL.Image and matplotlib.patches were imported but never used — removed
print("Imports OK.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Imports OK.


## 3. Configuration

In [3]:
# ── Model paths ──────────────────────────────────────────────
PLATE_MODEL_PATH   = "/content/best.pt"
VEHICLE_MODEL_NAME = "yolov8n.pt"

# ── COCO class IDs for target vehicles ───────────────────────
VEHICLE_CLASSES = {2: "Car", 3: "Motorcycle", 5: "Bus", 7: "Truck"}

# ── Detection thresholds ─────────────────────────────────────
VEHICLE_CONF_THRESHOLD = 0.40
PLATE_CONF_THRESHOLD   = 0.30

# ── Drawing colours (BGR) ────────────────────────────────────
BOX_COLORS = {
    "Car":        (0,   255,   0),
    "Motorcycle": (255, 165,   0),
    "Bus":        (0,   165, 255),
    "Truck":      (255,   0, 255),
}
PLATE_BOX_COLOR = (0, 0, 255)

# ── Video processing ─────────────────────────────────────────
BATCH_SIZE  = 4    # FIX: number of frames per YOLO GPU batch
FRAME_SKIP  = 2    # process 1-in-N frames for detection; annotate every frame

# ── OCR cache ────────────────────────────────────────────────
IOU_CACHE_THRESHOLD = 0.70   # FIX: reuse plate text when bbox IoU >= this value

# ── Plate upscale ────────────────────────────────────────────
OCR_TARGET_HEIGHT = 64   # FIX: target height in pixels before OCR
OCR_MAX_SCALE     = 4    # FIX: hard cap on upscale factor

# ── Output paths ─────────────────────────────────────────────
OUTPUT_IMAGE_PATH = "/content/annotated_image.jpg"
OUTPUT_VIDEO_RAW  = "/content/annotated_video_raw.mp4"   # mp4v write
OUTPUT_VIDEO_PATH = "/content/annotated_video.mp4"       # H.264 re-encode

# ─────────────────────────────────────────────────────────────
# FIX: Build COLOR_RANGES ONCE here with np.array() already applied.
# In v1 this dict was rebuilt inside detect_vehicle_color() on every call.
# ─────────────────────────────────────────────────────────────
COLOR_RANGES = {
    "Red":    [(np.array([0,   70,  50]), np.array([10,  255, 255])),
               (np.array([170, 70,  50]), np.array([180, 255, 255]))],
    "Orange": [(np.array([11,  70,  50]), np.array([25,  255, 255]))],
    "Yellow": [(np.array([26,  70,  50]), np.array([34,  255, 255]))],
    "Green":  [(np.array([35,  40,  40]), np.array([85,  255, 255]))],
    "Blue":   [(np.array([86,  50,  50]), np.array([130, 255, 255]))],
    "Brown":  [(np.array([10,  50,  20]), np.array([20,  200, 130]))],
}

print("Configuration ready.")
print(f"  Plate model  : {PLATE_MODEL_PATH}")
print(f"  Vehicle model: {VEHICLE_MODEL_NAME}")
print(f"  Batch size   : {BATCH_SIZE}  |  Frame skip: {FRAME_SKIP}")

Configuration ready.
  Plate model  : /content/best.pt
  Vehicle model: yolov8n.pt
  Batch size   : 4  |  Frame skip: 2


## 4. Load Models

In [4]:
# Vehicle detection model (YOLOv8n, COCO pretrained)
print("Loading YOLOv8 vehicle model...")
vehicle_model = YOLO(VEHICLE_MODEL_NAME)
print(f"  Vehicle model loaded: {VEHICLE_MODEL_NAME}")

# Custom number plate model
print("Loading custom plate model...")
try:
    if not os.path.exists(PLATE_MODEL_PATH):
        raise FileNotFoundError(
            f"Plate model not found at '{PLATE_MODEL_PATH}'.\n"
            "Upload best.pt to /content/ then re-run this cell."
        )
    plate_model = YOLO(PLATE_MODEL_PATH)
    PLATE_MODEL_AVAILABLE = True
    print(f"  Plate model loaded: {PLATE_MODEL_PATH}")
except FileNotFoundError as e:
    print(f"  [WARNING] {e}")
    plate_model = None
    PLATE_MODEL_AVAILABLE = False

# EasyOCR (GPU)
print("Initialising EasyOCR...")
ocr_reader = easyocr.Reader(['en'], gpu=True, verbose=False)
print("  EasyOCR ready (GPU).")

Loading YOLOv8 vehicle model...


  Vehicle model loaded: yolov8n.pt
Loading custom plate model...
  [WARNING] Plate model not found at '/content/best.pt'.
Upload best.pt to /content/ then re-run this cell.
Initialising EasyOCR...


  EasyOCR ready (GPU).


## 5. Helper — IoU for OCR Cache

In [5]:
def _iou(a: tuple, b: tuple) -> float:
    """
    Compute Intersection-over-Union between two bboxes (x1,y1,x2,y2).
    Used by the OCR cache to decide whether a vehicle is the same one
    seen in the previous processed frame.
    """
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1); iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2); iy2 = min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0:
        return 0.0
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    return inter / (area_a + area_b - inter)

print("IoU helper defined.")

IoU helper defined.


## 6. Core Detection Functions

In [6]:
#  FUNCTION 1 – detect_vehicles()
def detect_vehicles(frame: np.ndarray) -> list:
    """
    Run YOLOv8 on one BGR frame; return only target vehicle classes.

    Returns list of dicts: {bbox, class_name, confidence}
    """
    detections = []
    try:
        results = vehicle_model(frame, conf=VEHICLE_CONF_THRESHOLD,
                                verbose=False)[0]
        for box in results.boxes:
            cls_id = int(box.cls[0])
            if cls_id not in VEHICLE_CLASSES:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            detections.append({
                "bbox":       (x1, y1, x2, y2),
                "class_name": VEHICLE_CLASSES[cls_id],
                "confidence": float(box.conf[0]),
            })
    except Exception as e:
        print(f"[ERROR] detect_vehicles: {e}")
    return detections

#  FUNCTION 2 – detect_vehicle_color()

def detect_vehicle_color(frame: np.ndarray, bbox: tuple) -> str:
    """
    Classify dominant vehicle colour from the upper-60 % of its crop.

    FIX v1→v2: COLOR_RANGES is now a module-level constant with
    np.array() already applied — zero per-call allocation overhead.
    """
    try:
        x1, y1, x2, y2 = bbox
        crop_h = max(1, int((y2 - y1) * 0.6))
        crop = frame[y1 : y1 + crop_h, x1 : x2]
        if crop.size == 0:
            return "Unknown"

        hsv      = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        mean_sat = float(np.mean(hsv[:, :, 1]))
        mean_val = float(np.mean(hsv[:, :, 2]))

        #  Achromatic fast-path
        if mean_sat < 40:
            if   mean_val > 200: return "White"
            elif mean_val < 60:  return "Black"
            elif mean_val > 150: return "Silver"
            else:                return "Gray"

        # ── Chromatic scan using pre-built numpy arrays ───────
        best_color, best_count = "Unknown", 0
        total_pixels = crop.shape[0] * crop.shape[1]
        for color_name, ranges in COLOR_RANGES.items():
            count = sum(
                int(np.count_nonzero(cv2.inRange(hsv, lo, hi)))
                for lo, hi in ranges
            )
            if count > best_count:
                best_count = count
                best_color = color_name

        if best_count < total_pixels * 0.05:
            return "Silver" if mean_val > 150 else "Gray"

        return best_color

    except Exception as e:
        print(f"[ERROR] detect_vehicle_color: {e}")
        return "Unknown

#  FUNCTION 3 – detect_number_plate()
def detect_number_plate(frame: np.ndarray, vehicle_bbox: tuple):
    """
    Run the custom plate model inside the vehicle crop.

    Returns ((px1,py1,px2,py2), plate_crop) in absolute coords, or None.
    """
    if not PLATE_MODEL_AVAILABLE:
        return None
    try:
        vx1, vy1, vx2, vy2 = vehicle_bbox
        crop = frame[vy1:vy2, vx1:vx2]
        if crop.size == 0:
            return None
res = plate_model(crop, conf=PLATE_CONF_THRESHOLD, verbose=False)[0]
        if not len(res.boxes):
            return None
# highest-confidence box only
        best = max(res.boxes, key=lambda b: float(b.conf[0]))
        px1, py1, px2, py2 = map(int, best.xyxy[0].tolist())
        plate_crop = crop[py1:py2, px1:px2]
        if plate_crop.size == 0:
            return None
abs_bbox = (vx1 + px1, vy1 + py1, vx1 + px2, vy1 + py2)
        return abs_bbox, plate_crop
         except Exception as e:
        print(f"[ERROR] detect_number_plate: {e}")
        return None

#  FUNCTION 4 – recognize_plate_text()
def recognize_plate_text(plate_crop: np.ndarray) -> str:
    """
    Pre-process plate crop and run EasyOCR.

    FIX v1→v2:
      Old formula: scale = max(1, 100 // min(shape))  → fx = scale*2
        Problem  : a 10-px tall crop gave scale=10 → fx=20 (20× upscale),
                   producing a massive image that slows OCR and hurts accuracy.
      New formula: scale to OCR_TARGET_HEIGHT (64 px), capped at OCR_MAX_SCALE (4×).
    """
    try:
        if plate_crop is None or plate_crop.size == 0:
            return ""

        h, w = plate_crop.shape[:2]
        if h == 0 or w == 0:
            return ""
        # Controlled upscale
        scale  = min(OCR_TARGET_HEIGHT / h, float(OCR_MAX_SCALE))
        new_h  = max(int(h * scale), 1)
        new_w  = max(int(w * scale), 1)
        resized = cv2.resize(plate_crop, (new_w, new_h),
                             interpolation=cv2.INTER_CUBIC)

        # Pre-processing: grayscale → denoise → Otsu threshold
        gray  = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
        blur  = cv2.GaussianBlur(gray, (3, 3), 0)
        _, thresh = cv2.threshold(blur, 0, 255,
                                  cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        allowlist = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-'
        ocr_res   = ocr_reader.readtext(thresh, detail=1,
                                        allowlist=allowlist)
# Fallback to raw crop if threshold gave nothing
        if not ocr_res:
            ocr_res = ocr_reader.readtext(resized, detail=1,
                                          allowlist=allowlist)
        if not ocr_res:
            return ""

        texts   = [r[1].upper().strip() for r in ocr_res if r[2] > 0.2]
        cleaned = re.sub(r'[^A-Z0-9\-]', '',
                         ' '.join(texts).replace(' ', '-'))
        return cleaned

    except Exception as e:
        print(f"[ERROR] recognize_plate_text: {e}")
        return ""
print("Detection functions defined.")

Detection functions defined.


## 7. Annotation Function

In [7]:
#  FUNCTION 5 – annotate_results()
# Constants hoisted out — computed once, shared across every call
_FONT       = cv2.FONT_HERSHEY_SIMPLEX
_FONT_SCALE = 0.55
_THICKNESS  = 1

def annotate_results(frame: np.ndarray, vehicle_results: list) -> np.ndarray:
    """
    Draw bounding boxes and labels on a copy of `frame`.

    Label format: VehicleType | Color | PlateText

    FIX v1→v2:
      - cv2.getTextSize called ONCE per vehicle (was called twice).
      - Font constants hoisted to module level (no re-assignment per call).
    """
    out = frame.copy()

    for veh in vehicle_results:
        x1, y1, x2, y2 = veh["bbox"]
        cls   = veh.get("class_name", "Vehicle")
        color = veh.get("color",      "Unknown")
        plate = veh.get("plate_text", "") or "No Plate"
        box_c = BOX_COLORS.get(cls, (0, 255, 0))

        # Vehicle bounding box
        cv2.rectangle(out, (x1, y1), (x2, y2), box_c, 2)

        # Label (single getTextSize call)
        label = f"{cls} | {color} | {plate}"
        (tw, th), _ = cv2.getTextSize(label, _FONT, _FONT_SCALE, _THICKNESS)
        lx = x1
        ly = max(y1 - 6, th + 6)          # keep label inside frame top
        cv2.rectangle(out, (lx, ly - th - 4), (lx + tw + 4, ly + 2), box_c, -1)
        cv2.putText(out, label, (lx + 2, ly - 2),
                    _FONT, _FONT_SCALE, (0, 0, 0), _THICKNESS, cv2.LINE_AA)

        # Plate bounding box + text
        if veh.get("plate_bbox"):
            px1, py1, px2, py2 = veh["plate_bbox"]
            cv2.rectangle(out, (px1, py1), (px2, py2), PLATE_BOX_COLOR, 2)
            if plate != "No Plate":
                (ptw, pth), _ = cv2.getTextSize(plate, _FONT, 0.5, 1)
                cv2.rectangle(out, (px1, py2),
                              (px1 + ptw + 4, py2 + pth + 6),
                              PLATE_BOX_COLOR, -1)
                cv2.putText(out, plate, (px1 + 2, py2 + pth + 2),
                            _FONT, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

    return out
print("annotate_results defined.")

annotate_results defined.


## 8. Image Pipeline

In [8]:

#  FUNCTION 6 – process_image()
def process_image(image_path: str) -> list:
    """
    Full pipeline for a single image:
      load → detect vehicles → colour → plate detect → OCR
      → annotate → save → display side-by-side

    Returns list of result dicts.
    """
    # Load
    try:
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")
        frame = cv2.imread(image_path)
        if frame is None:
            raise ValueError(f"cv2.imread returned None for: {image_path}")
    except Exception as e:
        print(f"[ERROR] process_image: {e}")
        return []

    print(f"\nImage  : {image_path}  ({frame.shape[1]}×{frame.shape[0]})")

    #  Detect vehicles
    detections = detect_vehicles(frame)
    if not detections:
        print("[INFO] No vehicles detected.")
        return []
    print(f"[INFO] {len(detections)} vehicle(s) found.")

    #  Per-vehicle analysis
    vehicle_results = []
    for idx, det in enumerate(detections, 1):
        bbox = det["bbox"]
        cls  = det["class_name"]

        color = detect_vehicle_color(frame, bbox)

        plate_result = detect_number_plate(frame, bbox)
        plate_bbox, plate_text = None, ""
        if plate_result:
            plate_bbox, plate_crop = plate_result
            plate_text = recognize_plate_text(plate_crop)
            if not plate_text:
                print(f"  [{idx}] {cls}: plate detected but OCR returned empty.")
        else:
            msg = "plate model not loaded" if not PLATE_MODEL_AVAILABLE \
                  else "no plate detected"
            print(f"  [{idx}] {cls}: {msg}.")

        r = {
            "bbox":       bbox,
            "class_name": cls,
            "confidence": det["confidence"],
            "color":      color,
            "plate_bbox": plate_bbox,
            "plate_text": plate_text,
        }
        vehicle_results.append(r)
        plate_disp = plate_text if plate_text else "N/A"
        print(f"  [{idx}] {cls} | {color} | {plate_disp}  "
              f"(conf {det['confidence']:.2f})")

    # Annotate & save
    annotated = annotate_results(frame, vehicle_results)
    cv2.imwrite(OUTPUT_IMAGE_PATH, annotated)
    print(f"\n[SAVED] {OUTPUT_IMAGE_PATH}")

    # Display side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    axes[0].imshow(cv2.cvtColor(frame,     cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original",   fontsize=14); axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Annotated",  fontsize=14); axes[1].axis('off')
    plt.suptitle("Vehicle ANPR — Result", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return vehicle_results


print("process_image defined.")

process_image defined.


## 9. Video Pipeline

In [9]:

#  FUNCTION 7 – process_video()
def process_video(video_path: str,
                  frame_skip: int = FRAME_SKIP,
                  batch_size: int = BATCH_SIZE) -> str:
    """
    Process a video file with batched YOLO inference + OCR caching.

    Key optimisations vs v1
    ───────────────────────
    1. BATCHED YOLO  : `batch_size` frames are stacked and sent to the
       GPU in one call instead of one-by-one.

    2. OCR CACHE     : Each detected vehicle is compared against the
       previous frame's detections via IoU. If IoU ≥ IOU_CACHE_THRESHOLD
       the cached plate text is reused — EasyOCR is not called again.
       This is the single biggest speed-up for video: OCR is ~200-500 ms
       per plate; caching can cut total OCR calls by 80-90 %.

    3. H.264 RE-ENCODE: After writing with mp4v (required by OpenCV),
       ffmpeg re-encodes to H.264 so the Colab Video() widget plays it.

    Args:
        video_path : input video path
        frame_skip : run detection every N frames; annotate every frame
        batch_size : frames per YOLO batch

    Returns:
        str: path to the H.264 output video
    """
    # Open video
    try:
        if not os.path.exists(video_path):
            raise FileNotFoundError(f"Video not found: {video_path}")
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Cannot open video: {video_path}")
    except Exception as e:
        print(f"[ERROR] process_video: {e}")
        return ""

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"\nVideo  : {video_path}")
    print(f"  {width}×{height}  |  {fps:.1f} fps  |  {total} frames")
    print(f"  batch_size={batch_size}  frame_skip={frame_skip}")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(OUTPUT_VIDEO_RAW, fourcc, fps, (width, height))

    # State
    frame_idx    = 0
    last_results = []      # results from the last detection pass
    ocr_cache    = []      # list of {bbox, plate_text} from last pass

    # batch accumulation
    batch_frames  = []     # raw frames in current batch
    batch_indices = []     # frame index for each batch entry
    pending_write = {}     # frame_idx → annotated frame (awaiting writer)

    start = time.time()

    def _run_batch(frames_batch: list, indices_batch: list):
        """
        Run YOLO on a batch of frames, then colour + plate + OCR per vehicle.
        Updates last_results and ocr_cache in outer scope.
        Returns dict: frame_idx → annotated frame.
        """
        nonlocal last_results, ocr_cache

        annotated_map = {}

        #  Single batched YOLO call
        try:
            batch_results = vehicle_model(
                frames_batch,
                conf=VEHICLE_CONF_THRESHOLD,
                verbose=False
            )
        except Exception as e:
            print(f"[ERROR] batched YOLO: {e}")
            for i, fr in zip(indices_batch, frames_batch):
                annotated_map[i] = fr
            return annotated_map

        for frame_res, frame_bgr, fidx in zip(
                batch_results, frames_batch, indices_batch):

            frame_detections = []
            new_ocr_cache    = []

            for box in frame_res.boxes:
                cls_id = int(box.cls[0])
                if cls_id not in VEHICLE_CLASSES:
                    continue
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                bbox  = (x1, y1, x2, y2)
                cls   = VEHICLE_CLASSES[cls_id]
                conf  = float(box.conf[0])
                color = detect_vehicle_color(frame_bgr, bbox)

                # ── OCR CACHE LOOKUP
                # Check if this vehicle overlaps a cached entry
                plate_bbox, plate_text = None, ""
                cached_hit = False
                for cached in ocr_cache:
                    if _iou(bbox, cached["bbox"]) >= IOU_CACHE_THRESHOLD:
                        plate_text = cached["plate_text"]
                        plate_bbox = cached.get("plate_bbox")
                        cached_hit = True
                        break

                # ── Run plate detection + OCR only on cache miss
                if not cached_hit:
                    pr = detect_number_plate(frame_bgr, bbox)
                    if pr:
                        plate_bbox, plate_crop = pr
                        plate_text = recognize_plate_text(plate_crop)

                new_ocr_cache.append({
                    "bbox":       bbox,
                    "plate_bbox": plate_bbox,
                    "plate_text": plate_text,
                })
                frame_detections.append({
                    "bbox":       bbox,
                    "class_name": cls,
                    "confidence": conf,
                    "color":      color,
                    "plate_bbox": plate_bbox,
                    "plate_text": plate_text,
                })

            last_results = frame_detections
            ocr_cache    = new_ocr_cache
            annotated_map[fidx] = annotate_results(frame_bgr, frame_detections)

        return annotated_map

    # ── Main read loop
    while True:
        ret, frame = cap.read()
        if not ret:
            # flush remaining batch
            if batch_frames:
                ann_map = _run_batch(batch_frames, batch_indices)
                for i in sorted(ann_map):
                    writer.write(ann_map[i])
            break

        if frame_idx % frame_skip == 0:
            # accumulate into batch
            batch_frames.append(frame)
            batch_indices.append(frame_idx)

            if len(batch_frames) >= batch_size:
                ann_map = _run_batch(batch_frames, batch_indices)
                for i in sorted(ann_map):
                    writer.write(ann_map[i])
                batch_frames.clear()
                batch_indices.clear()
        else:
            # skipped frame → re-use last annotations
            writer.write(annotate_results(frame, last_results))

        frame_idx += 1
        if frame_idx % 100 == 0:
            pct = (frame_idx / total * 100) if total else 0
            print(f"  {frame_idx}/{total} frames  ({pct:.1f}%)  "
                  f"[{time.time()-start:.1f}s]")

    cap.release()
    writer.release()

    elapsed = time.time() - start
    print(f"\n[INFO] Processing done in {elapsed:.1f}s  "
          f"({frame_idx} frames, ~{frame_idx/elapsed:.1f} fps effective)")

    # ── FIX: re-encode mp4v → H.264 so Colab Video() plays it ─
    print("Re-encoding to H.264 for Colab playback...")
    os.system(
        f"ffmpeg -y -i {OUTPUT_VIDEO_RAW} "
        f"-vcodec libx264 -acodec aac "
        f"-loglevel error {OUTPUT_VIDEO_PATH}"
    )
    if os.path.exists(OUTPUT_VIDEO_PATH):
        print(f"[SAVED] {OUTPUT_VIDEO_PATH}")
    else:
        print("[WARNING] H.264 encode failed; raw mp4v file kept.")
        return OUTPUT_VIDEO_RAW

    return OUTPUT_VIDEO_PATH


print("process_video defined.")

process_video defined.


## 10. Upload Plate Model (Optional)

In [ ]:
print("Upload an image (JPG / PNG / BMP)...")
try:
    up = files.upload()
    if not up:
        raise ValueError("No file uploaded.")
    fn = list(up.keys())[0]
    img_path = f"/content/{fn}"
    with open(img_path, 'wb') as f:
        f.write(up[fn])
    print(f"Uploaded: {img_path}")
except Exception as e:
    print(f"[ERROR] {e}")
    img_path = None

In [ ]:
if img_path and os.path.exists(img_path):
    image_results = process_image(img_path)

    if image_results:
        print("\n" + "="*58)
        print("  DETECTION SUMMARY")
        print("="*58)
        print(f"  {'#':<4} {'Type':<14} {'Color':<10} {'Plate':<18} {'Conf'}")
        print("-"*58)
        for i, r in enumerate(image_results, 1):
            pt = r['plate_text'] or 'N/A'
            print(f"  {i:<4} {r['class_name']:<14} {r['color']:<10} "
                  f"{pt:<18} {r['confidence']:.2f}")
        print("="*58)

    if os.path.exists(OUTPUT_IMAGE_PATH):
        files.download(OUTPUT_IMAGE_PATH)
else:
    print("[ERROR] No valid image. Please re-upload.")

## 12. Run — Video

In [11]:
print("Upload a video (MP4 / AVI / MOV)...")
try:
    up = files.upload()
    if not up:
        raise ValueError("No file uploaded.")
    fn = list(up.keys())[0]
    vid_path = f"/content/{fn}"
    with open(vid_path, 'wb') as f:
        f.write(up[fn])
    print(f"Uploaded: {vid_path}")
except Exception as e:
    print(f"[ERROR] {e}")
    vid_path = None

Upload a video (MP4 / AVI / MOV)...


Saving 2103099-hd_1920_1080_30fps.mp4 to 2103099-hd_1920_1080_30fps.mp4
Uploaded: /content/2103099-hd_1920_1080_30fps.mp4


In [ ]:
if vid_path and os.path.exists(vid_path):
    out_vid = process_video(vid_path)

    if out_vid and os.path.exists(out_vid):
        print("\nVideo preview:")
        display(Video(out_vid, embed=True, width=800))
        files.download(out_vid)
else:
    print("[ERROR] No valid video. Please re-upload.")

## 13. Quick Test (No Upload)

In [ ]:
import urllib.request

SAMPLE_URL  = "https://ultralytics.com/images/bus.jpg"
SAMPLE_PATH = "/content/sample_test.jpg"

try:
    urllib.request.urlretrieve(SAMPLE_URL, SAMPLE_PATH)
    print(f"Sample downloaded → {SAMPLE_PATH}")
    process_image(SAMPLE_PATH)
except Exception as e:
    print(f"[ERROR] {e}")